In [0]:

# df = spark.read.option("header", "true").format("csv").load("abfss://unitycatalog@bhavanmetadata.dfs.core.windows.net/raw/csut12.csv")
# df.show()
# df.write.saveAsTable("ram.default.csut12")

In [0]:
yourfile='data_f0051149-9530-4309-8145-ee2f3bcaf5e4_86665c09-99b2-4e53-852b-1f94b78b1b7f.txt'
df = spark.read.option("header", "true").format("csv").load(f"abfss://raw@bhavanmetadata.dfs.core.windows.net/{yourfile}")
df.show(truncate=True)
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ram.default.todayjob")

In [0]:
https://bhavanmetadata.blob.core.windows.net/raw/data_f0051149-9530-4309-8145-ee2f3bcaf5e4_86665c09-99b2-4e53-852b-1f94b78b1b7f.txt

In [0]:
%sql
describe extended ram.default.csut12

In [0]:
df = spark.read.option("header", "true").format("csv").load("abfss://raw@bhavanmetadata.dfs.core.windows.net/csut12.csv")
df.show()

In [0]:
%sql
SELECT * FROM ram.default.source_view;

In [0]:
source_data = [
    (1, "Arun", "Chennai"),
    (2, "Ravi", "Hyderabad"),
    (3, "Meena", "Mumbai")
]

source_df = spark.createDataFrame(source_data, ["customer_id", "name", "city"])
from pyspark.sql.functions import current_date, lit
from pyspark.sql.functions import lit, current_date

source_scd2_df = source_df \
    .withColumn("is_current", lit("Y")) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))
source_scd2_df.write.format('delta').options(overwrite='true', overwriteSchema='true').saveAsTable("ram.default.source_view")


In [0]:
%sql
select * from ram.default.customer_target

In [0]:
%sql
select * from ram.default.customer_target

In [0]:
target_data = [
    (1, "Arun", "Delhi", "2024-01-01", "9999-12-31", "Y"),
    (2, "Ravi", "Hyderabad", "2024-01-01", "9999-12-31", "Y"),
    (4, "John", "Pune", "2024-01-01", "9999-12-31", "Y")
]

target_df = spark.createDataFrame(
    target_data,
    ["customer_id", "name", "city", "start_date", "end_date", "is_current"]
)
target_df.write.format('delta').options(overwriteSchema='true').saveAsTable("ram.default.customer_target")

In [0]:
from pyspark.sql.functions import current_date, lit
from pyspark.sql.functions import lit, current_date

source_scd2_df = source_df \
    .withColumn("is_current", lit("Y")) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

In [0]:
%sql
MERGE INTO ram.default.customer_target t
USING ram.default.source_view s
ON t.customer_id = s.customer_id
AND t.is_current = 'Y'

WHEN MATCHED AND (
    t.name <> s.name OR
    t.city <> s.city
)
THEN UPDATE SET
    t.end_date = current_date(),
    t.is_current = 'N'

In [0]:
%sql
INSERT INTO ram.default.customer_target
SELECT
    s.customer_id,
    s.name,
    s.city,
    s.start_date,
    s.end_date,
    s.is_current
FROM ram.default.source_view s
LEFT JOIN ram.default.customer_target t
ON s.customer_id = t.customer_id
AND t.is_current = 'Y'
WHERE t.customer_id IS NULL
   OR t.name <> s.name
   OR t.city <> s.city;

In [0]:
# Sample source data
source_data = [
    (1, "Arun", "Chennai"),
    (2, "Ravi", "Hyderabad"),
    (3, "Meena", "Mumbai"),
    (4, "John", "Pune")
]
source_df = spark.createDataFrame(source_data, ["customer_id", "name", "city"])

# Prepare SCD2 source view
from pyspark.sql.functions import current_date, lit
source_scd2_df = source_df \
    .withColumn("is_current", lit("Y")) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

source_scd2_df.write.format('delta').mode('overwrite').saveAsTable("ram.default.source_view")

# Sample target table with historical data
target_data = [
    (1, "Arun", "Delhi", "2024-01-01", "9999-12-31", "Y"),
    (2, "Ravi", "Hyderabad", "2024-01-01", "9999-12-31", "Y"),
    (4, "John", "Pune", "2024-01-01", "9999-12-31", "Y")
]
target_df = spark.createDataFrame(
    target_data,
    ["customer_id", "name", "city", "start_date", "end_date", "is_current"]
)
target_df.write.format('delta').mode('overwrite').saveAsTable("ram.default.customer_target")

# SCD2 Merge: Close old records and insert new/changed records
merge_sql = """
MERGE INTO ram.default.customer_target t
USING ram.default.source_view s
ON t.customer_id = s.customer_id AND t.is_current = 'Y'
WHEN MATCHED AND (t.name <> s.name OR t.city <> s.city)
  THEN UPDATE SET t.end_date = current_date(), t.is_current = 'N'
WHEN NOT MATCHED
  THEN INSERT (customer_id, name, city, start_date, end_date, is_current)
       VALUES (s.customer_id, s.name, s.city, s.start_date, s.end_date, s.is_current)
"""
spark.sql(merge_sql)

# Display final SCD2 table
display(spark.table("ram.default.customer_target"))